# 06. Experimentos iterativos

Permite comparar rápidamente nuevas combinaciones de features manteniendo el mismo protocolo de entrenamiento y evaluación.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "outputs" / "plots"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

for directory in [DATA_PROCESSED_DIR, MODELS_DIR, PLOTS_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.evaluation import evaluate_model
from src.feature_engineering import DEFAULT_FEATURE_COLUMNS
from src.modeling import train_xgboost_model


In [ ]:
features_df = pd.read_parquet(DATA_PROCESSED_DIR / "features.parquet")

feature_sets = {
    "baseline_original": DEFAULT_FEATURE_COLUMNS,
    "sin_temporales": [
        column
        for column in DEFAULT_FEATURE_COLUMNS
        if column not in {"mes", "dia_semana", "fin_mes"}
    ],
    "sin_stats_bus": [
        column
        for column in DEFAULT_FEATURE_COLUMNS
        if not column.startswith("dias_desde_correctivo_anterior_")
        and column != "correctivos_previos_max"
    ],
    "solo_historial_y_tiempo": [
        "dias_desde_correctivo_anterior",
        "correctivos_previos",
        "correctivos_ult_7d",
        "correctivos_ult_5d",
        "correctivos_ult_3d",
        "mes",
        "dia_semana",
        "fin_mes",
    ],
}

EXPERIMENT_USE_SMOTE = True
EXPERIMENT_THRESHOLD = 0.5
experiment_rows = []

for window in [7, 5, 3]:
    y = features_df[f"correctivo_prox_{window}d"].astype(int)

    for experiment_name, columns in feature_sets.items():
        X = features_df[columns].fillna(0)
        training_artifacts = train_xgboost_model(X, y, use_smote=EXPERIMENT_USE_SMOTE)
        metrics = evaluate_model(
            training_artifacts["y_test"],
            training_artifacts["y_score"],
            thresholds=(EXPERIMENT_THRESHOLD,),
        )
        report = metrics["threshold_metrics"][str(EXPERIMENT_THRESHOLD)]["classification_report"]
        positive_metrics = report.get("1", {})
        experiment_rows.append(
            {
                "experiment_name": experiment_name,
                "window_days": window,
                "use_smote": EXPERIMENT_USE_SMOTE,
                "threshold": EXPERIMENT_THRESHOLD,
                "feature_count": len(columns),
                "precision_class_1": positive_metrics.get("precision"),
                "recall_class_1": positive_metrics.get("recall"),
                "f1_class_1": positive_metrics.get("f1-score"),
                "scale_pos_weight": training_artifacts["scale_pos_weight"],
            }
        )

experiments_df = pd.DataFrame(experiment_rows).sort_values(
    ["window_days", "f1_class_1"],
    ascending=[True, False],
)
experiments_path = METRICS_DIR / "experimentos_iterativos.csv"
experiments_df.to_csv(experiments_path, index=False)

display(experiments_df)
print("Resultados guardados en:", experiments_path)


In [ ]:
for window in [7, 5, 3]:
    subset = experiments_df[experiments_df["window_days"] == window]
    plt.figure(figsize=(8, 4))
    plt.bar(subset["experiment_name"], subset["f1_class_1"])
    plt.title(f"F1 clase positiva - horizonte {window} días")
    plt.ylabel("F1")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"experimentos_f1_{window}d.png", dpi=150)
    plt.show()
